# Basic workflow for making a "surrogate" model that learns the beam-spin asymmetry as function of $x_{\text{B}}$, $t$, $Q^{2}$, and $\phi$.

## (1): Import Libraries

### (1.1): Import Native Libraries:

In [ ]:
import datetime
import gc
from pathlib import Path

### (1.2): Import 3rd-Party Libraries:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

### (1.3): Library Versions:

In [ ]:
print(f"[INFO]: numpy version: {np.__version__}")
print(f"[INFO]: pandas version: {pd.__version__}")
print(f"[INFO]: tensorflow version: {tf.__version__}")

### (1.4): Versioning:

In [ ]:
VERSION_NUMBER = 1
MINOR_NUMBER = 1
MAJOR_MINOR_NUMBER = f"{VERSION_NUMBER}_{MINOR_NUMBER}"

print(f"[INFO]: We are saving figures and data with the following appendage: {MAJOR_MINOR_NUMBER}")

### (1.5): Program Parameters *and* DNN Hyperparameters:

In [ ]:
# do you want to standardized *both* the x- and y-data?
STANDARDIZING_DATA = True

# DNN hyperparameters:
NUMBER_OF_REPLICAS = 10
BASE_LEARNING_RATE = 3e-4
BASE_WEIGHT_DECAY_RATE = 1e-7
TOTAL_EPOCHS = 1500
SYMMETRY_LOSS_PARAMETER = 0.0

# train/validation/test split:
_DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE = 0.1 # 90% temporary, 10% testing
_DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE = 0.1 # of the above 90% temporary, 90% training, 10% validation

USING_GAUSSIAN_ERROR_SAMPLING = False

# setting the TF global seed
tf.random.set_seed(31415926535)

### (1.6): Pathing:

In [ ]:
output_directory = Path("./local")

## (2): Plotting Styles:

In [ ]:
plt.rcParams.update({"text.usetex": True, "font.family": "serif"})
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['xtick.major.size'] = 8.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['xtick.minor.size'] = 3.5
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['xtick.top'] = True
plt.rcParams['xtick.labelsize'] = 15
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['ytick.major.size'] = 8.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['ytick.minor.size'] = 3.5
plt.rcParams['ytick.minor.width'] = 0.5
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['savefig.dpi'] = 300

## (3): Data Loading:

### (3.1): Loading Main File:

In [ ]:
test_dataframe = pd.read_csv(
    filepath_or_buffer =
        output_directory /
        f"version_{MAJOR_MINOR_NUMBER}" /
        "data" /
        f"refined_bsa_data_v{MAJOR_MINOR_NUMBER}.csv"
)

### (3.2): Loading in the supervised learning $(x, y)$ pairs:

In [ ]:
# saves a copy of the column:
test_dataframe['original_bsa'] = test_dataframe['unp_target_bsa']

if USING_GAUSSIAN_ERROR_SAMPLING:

    test_dataframe['unp_target_bsa'] = np.random.normal(
        loc = test_dataframe['original_bsa'],
        scale = test_dataframe['unp_target_bsa_err']
    )

# phi -> v(phi)
test_dataframe["v"] = np.sin(test_dataframe["phi"])

x_data = test_dataframe[["k", "q_squared", "x_b", "t", "v"]]
# [NOTE]: we do NOT LOG THE BSA DATA!
y_data = test_dataframe[["unp_target_bsa"]]

TOTAL_DATA_SIZE = len(x_data)
print(f"[INFO]: Total data size is: {TOTAL_DATA_SIZE}")

### (3.3): Data Preprocessing:

In [ ]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

In [ ]:
if STANDARDIZING_DATA:
    print("[INFO]: We are standardizing the data.")
    preprocessed_x_data = x_scaler.fit_transform(x_data)
    preprocessed_y_data = y_scaler.fit_transform(y_data)
else:
    print("[INFO]: We are not standardizing the data here.")
    preprocessed_x_data = x_data
    preprocessed_y_data = y_data

### (3.5): Splitting along training/validation/testing:

In [ ]:
x_remaining, x_testing, y_remaining, y_testing = train_test_split(
    x_data, y_data,
    test_size = _DNN_TESTING_TEMPORARY_SPLIT_PERCENTAGE, shuffle = True, random_state = 31415)

x_training, x_validation, y_training, y_validation = train_test_split(
    x_remaining, y_remaining,
    test_size = _DNN_TRAINING_VALIDATION_SPLIT_PERCENTAGE, shuffle = True, random_state = 31415)

total_training_points = len(x_training)
total_validation_points = len(x_validation)
total_testing_points = len(x_testing)

print(f"[INFO]: Total training points is = {total_training_points}")
print(f"[INFO]: Total validation points is = {total_validation_points}")
print(f"[INFO]: Total testing points is = {total_testing_points}")

## (4): DNN Stuff:

### (4.2): DNN Architecture:

In [ ]:
class BSASurrogateModel(tf.keras.Model):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.hidden_layers = [
            tf.keras.layers.Dense(
                128, activation = "silu", kernel_initializer = "glorot_normal"
            )
            for _ in range(4)
        ]

        # linear activation is default activation if `activation` key is not specified: https://www.tensorflow.org/api_docs/python/tf/keras/layers/Dense
        self.bsa_output = tf.keras.layers.Dense(1)

    def call(self, x):

        # nothing fancy here!
        for layer in self.hidden_layers:
            x = layer(x)
        
        return self.bsa_output(x)

## (5): **Actually Fitting the Model**:

### (5.1): Select the batch size here:

In [ ]:
BATCH_SIZE = len(x_training)

### **(5.2): The Fit Routine!**

In [ ]:
all_histories = []
all_point_predictions = []
models = []

In [ ]:
for index in range(NUMBER_OF_REPLICAS):
    replica_number = index + 1
    print(f"[INFO]: Now training replica #{replica_number}")

    tf.keras.backend.clear_session()
    gc.collect()

    dnn_model = BSASurrogateModel()
    dnn_model.compile(
        # LR is alpha in ADAM, which is stepsize:
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate = BASE_LEARNING_RATE,
            weight_decay = BASE_WEIGHT_DECAY_RATE),
        loss = "mse",
        metrics = ["mae"])

    dnn_model_history = dnn_model.fit(
        x_training, y_training,
        validation_data = (x_validation, y_validation),
        epochs = TOTAL_EPOCHS,
        batch_size = BATCH_SIZE,
        verbose = 0)
    
    # models stored in memory...
    all_histories.append(dnn_model_history.history)

    model_testing_evaluation_metrics = dnn_model.evaluate(x_testing, y_testing, verbose = 0)
    print(f"[INFO]: Evaluation metrics are: {model_testing_evaluation_metrics}")

    dictionary_of_keras_metrics = dict(zip(dnn_model.metrics_names, model_testing_evaluation_metrics))
    model_testing_loss = dictionary_of_keras_metrics["loss"]
    print(f"[INFO]: Test loss for replica #{replica_number}: {model_testing_loss}")

    figure, axis = plt.subplots(1, 1, figsize = (6, 6))

    axis.plot(dnn_model_history.history['loss'],
        label = "Training Loss", color = 'orange', alpha = 0.6)
    axis.plot(dnn_model_history.history['val_loss'],
        label = "Validation Loss", color = 'purple', alpha = 0.6)

    axis.set_xlabel(r"Epoch", fontsize = 14.)
    axis.set_ylabel(r"Loss", fontsize = 14.)
    axis.set_title(
        f"BSA Surrogate Model\n"
        rf"(Testing = {model_testing_loss:.7f})",
        fontsize = 18.)
    axis.legend(fontsize = 14.)
    axis.grid(visible = False)

    axis.text(
        0.00, -0.11,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes, verticalalignment = 'top',  horizontalalignment = 'left', fontsize = 9.,)

    for extension in ['png', 'eps']:
        figure.savefig(
            fname =
                output_directory /
                f"version_{MAJOR_MINOR_NUMBER}" /
                "learning_curves" /
                f"bsa_surrogate_lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(figure)

    del figure
    del axis

    # log losses
    figure, axis = plt.subplots(1, 1, figsize = (6, 6))

    axis.plot(
        np.arange(0, TOTAL_EPOCHS, 1),
        dnn_model_history.history['loss'], linewidth = 1.0,
        label = "Log Training Loss", color = 'orange', alpha = 0.6)
    axis.plot(
        np.arange(0, TOTAL_EPOCHS, 1),
        dnn_model_history.history['val_loss'], linewidth = 1.0,
        label = "Log Validation Loss", color = 'purple', alpha = 0.6)

    axis.set_yscale("log")

    axis.set_xlabel(r"Epoch", fontsize = 14.)
    axis.set_ylabel(r"Log (Loss)", fontsize = 14.)
    axis.set_title(
        f"BSA Surrogate Model\n"
        rf"(Testing = {model_testing_loss:.7f})",
        fontsize = 18.)
    axis.legend(fontsize = 14.)
    axis.grid(visible = False)

    axis.text(
        0.00, -0.11,
        f"Figure rendered {datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}", 
        transform = axis.transAxes, verticalalignment = 'top',  horizontalalignment = 'left', fontsize = 9.,)

    for extension in ['png', 'eps']:
        figure.savefig(
            fname = output_directory / 
            f"version_{MAJOR_MINOR_NUMBER}" / 
            "learning_curves" /
            f"bsa_surrogate_log_lc_replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(figure)

    del figure
    del axis
    
    # make a copy of the entire original dataframe:
    prediction_dataframe = test_dataframe.copy()
    
    # make the predictions:
    predictions_z = dnn_model.predict(preprocessed_x_data, verbose = 0)
    predicted_bsa = y_scaler.inverse_transform(predictions_z)
    
    # add model cross-section prediction to the dataframe:
    prediction_dataframe["model_bsa"] = predicted_bsa

    # saves the prediction dataframe:
    prediction_dataframe.to_csv(
        output_directory /
        f"version_{MAJOR_MINOR_NUMBER}" / 
        "data" / 
        f"replica_{replica_number}_predictions.csv",
        index = False)

    # save the model:
    dnn_model.save(
        filepath =
        output_directory /
        f"version_{MAJOR_MINOR_NUMBER}" / 
        "replicas" / 
        f"replica_{replica_number}_v{MAJOR_MINOR_NUMBER}.keras",
    )

    # add to the arrays:
    all_point_predictions.append(predicted_bsa)
    all_histories.append(dnn_model_history.history)
    # the models are in memory...
    models.append(dnn_model)

    figure, axis = plt.subplots(1, 1, figsize = (6, 6))
    r_squared = r2_score(test_dataframe["unp_target_bsa"], predicted_bsa)

    axis.scatter(
        test_dataframe["unp_target_bsa"], predicted_bsa,
        s = 4.0, alpha = 0.6, color = "blue")
    
    # have to calculate the "farthest points" for plotting the line:
    minimum_bsa_value = min(
        predicted_bsa.min(),
        test_dataframe["unp_target_bsa"].min())

    maximum_bsa_value = max(
        predicted_bsa.max(),
        test_dataframe["unp_target_bsa"].max())

    axis.plot(
        [minimum_bsa_value, maximum_bsa_value],
        [minimum_bsa_value, maximum_bsa_value],
        color = "red", linestyle = "-", label = "Line of Perfect Fit")
    
    axis.set_xlabel("BSA Data", fontsize = 14.0)
    axis.set_ylabel("DNN Prediction", fontsize = 14.0)
    axis.set_title(f"Replica {replica_number} Performance\n$R^2 = {r_squared:.5f}$")

    for extension in ['png', 'eps']:
        figure.savefig(
            fname =
                output_directory /
                f"version_{MAJOR_MINOR_NUMBER}" /
                "plots" / 
                f"replica_{replica_number}_prediction_quality_v{MAJOR_MINOR_NUMBER}.{extension}",
            facecolor = 'white',
            transparent = False)

    plt.close(figure)

## (6): Making Plots of Every Local Fit to the Available Data:

### (6.0): Need this function...

In [ ]:
# have to go through the intermediate transformations:
def predict_bsa(model, x_dataframe, x_scaler, y_scaler):
    
    x_scaled = x_scaler.transform(x_dataframe)
    model_prediction_in_z = model.predict(x_scaled, verbose = 0)
    bsa = y_scaler.inverse_transform(model_prediction_in_z)
    return bsa

### (6.1): Group the Dataframe by the Kinematic Settings:

In [ ]:
grouped = test_dataframe.groupby(['k', 't', 'x_b', 'q_squared'])

### (6.2): Make the Statistical Distribution Plots at Every Kinematic Setting:

In [ ]:
all_point_predictions = np.array(all_point_predictions)
average_prediction = np.mean(all_point_predictions, axis = 0)
standard_dev_prediction = np.std(all_point_predictions, axis = 0)

# this is trento convention: -pi to pi:
phi_smooth = np.linspace(-np.pi, np.pi, 361)

# for printing stuff later:
special_phis = [0, np.pi/2., -np.pi/2., np.pi]

for (k_value, t_value, xb_value, qsquared_value), group in grouped:
    print(f"[INFO]: Processing k = {k_value}, t = {t_value}, xb = {xb_value}, Q2 = {qsquared_value}")

    group = group.sort_values("phi")

    smooth_dataframe = pd.DataFrame({
        "k": np.full_like(phi_smooth, k_value),
        "q_squared": np.full_like(phi_smooth, qsquared_value),
        "x_b": np.full_like(phi_smooth,xb_value),
        "t": np.full_like(phi_smooth, t_value),
        "v": np.sin(phi_smooth)
    })

    smooth_predictions_all = np.array([
        predict_bsa(model, smooth_dataframe, x_scaler, y_scaler)
        for model in models
    ])

    smooth_mean = np.mean(smooth_predictions_all, axis = 0)
    smooth_std = np.std(smooth_predictions_all, axis = 0)

    bsa_smooth_mean = smooth_mean[:, 0]
    bsa_smooth_std = smooth_std[:, 0]

    point_dataframe = pd.DataFrame({
        "k": group["k"],
        "q_squared": group["q_squared"],
        "x_b": group["x_b"],
        "t": group["t"],
        "v": np.sin(group["phi"])
    })

    point_predictions_all = np.array([
        predict_bsa(model, point_dataframe, x_scaler, y_scaler)
        for model in models
    ])

    point_mean = np.mean(point_predictions_all, axis = 0)
    point_std = np.std(point_predictions_all, axis = 0)

    bsa_predicted = point_mean[:, 0]
    # not actually used:
    bsa_predicted_stddev = point_std[:, 0]

    # these are experimental values:
    phi = group["phi"].to_numpy()
    bsa_error = group["unp_target_bsa_err"].to_numpy()
    bsa_actual = group["unp_target_bsa"].to_numpy()

    pulls = (bsa_actual - bsa_predicted) / bsa_error

    chi_squared = np.sum(pulls**2)

    chi2_per_point = chi_squared / len(phi)

    bsa_residuals = bsa_actual - bsa_predicted

    for phi_target in special_phis:
        phi_index = np.argmin(np.abs(phi_smooth - phi_target))
        phi_actual = phi_smooth[phi_index]
        sigma_value = bsa_smooth_std[phi_index]
        print(f"[INFO]: phi = {phi_actual:.3f}, uncertainty = {sigma_value:.6f}")

    residuals_figure, residuals_axes = plt.subplots(2, 1, figsize = (10, 8), sharex = 'col', layout = "tight")

    residuals_axes[1].text(
        -0.1, -0.1,
        fr"Figure rendered {datetime.datetime.now().strftime('%y%m%d-%H%M%S')}", 
        transform = residuals_axes[1].transAxes)

    residuals_axes[0].plot(phi_smooth, bsa_smooth_mean, color = 'red', lw = 2, label = rf'Replica Average ($N = {NUMBER_OF_REPLICAS}$)')
    residuals_axes[0].fill_between(
        phi_smooth, bsa_smooth_mean - bsa_smooth_std, bsa_smooth_mean + bsa_smooth_std,
        color = 'red', alpha = 0.3,
        label = r'$\sigma$ band')

    residuals_axes[0].errorbar(
        phi, bsa_actual, yerr = bsa_error,
        fmt = 'o', mfc = 'white', mec = 'black', ms = 5, ecolor = 'black', elinewidth = 1, capsize = 2, alpha = 0.8,
        label = 'Experimental Data')
    residuals_axes[0].set_ylabel(r"BSA", fontsize = 16.)
    residuals_axes[0].set_title(rf"BSA ($\chi^2/N = {chi2_per_point:.7f}$)", fontsize = 18.)
    residuals_axes[0].legend(fontsize = 14.)
    residuals_axes[0].grid(True, linestyle = ':', alpha = 0.6)

    residuals_axes[1].scatter(phi, bsa_residuals, color = 'blue', alpha = 0.6)
    residuals_axes[1].axhline(0, color = 'black', linestyle = '--')
    residuals_axes[1].set_xlabel(r"$\phi$ (radians)", fontsize = 16.)
    residuals_axes[1].set_title("Residuals", fontsize = 18.)
    residuals_axes[1].grid(True, linestyle = ':', alpha = 0.6)

    residuals_figure.suptitle(
        "Kinematic Setting:\n"
        rf"$k = {k_value}$, $t = {t_value}$, $x_\mathrm{{B}} = {xb_value}$, $Q^2 = {qsquared_value}$",
        fontsize = 16.
    )

    filename = (
        output_directory /
        f"version_{MAJOR_MINOR_NUMBER}" / 
        "plots" / 
        f"k{k_value:.3f}_t{t_value:.3f}_xb{xb_value:.3f}_q2{qsquared_value:.3f}_residuals"
    )

    for extension in ["png", "eps"]:
        residuals_figure.savefig(
            f"{filename}.{extension}",
            facecolor = "white", transparent = False)

    plt.close(residuals_figure)


### (6.3): Make smoothly-interpolated Cross-Section Predictions:

#### (6.3.1): Foliate the four-dimensional space with unique $k$ values:

In [ ]:
unique_k_values = test_dataframe.groupby(['k'])
print(f"[INFO]: there are {unique_k_values.ngroups} unique values for k.")

#### (6.3.2): For every $k$, determine how many unique $x_{\text{B}}$ and $Q^{2}$ pairs there are; the pairs determine how many surface plots *per* $k$ we make:

In [ ]:
for k_value, k_group in unique_k_values:
    xb_q2_groups = k_group.groupby(['x_b', 'q_squared'])
    xb_q2_combinations = xb_q2_groups.ngroups
    print(f"[INFO]: found a total of {xb_q2_combinations} (xb, Q^2) values at k = {k_value}")

#### (6.3.3): Begin the plotting loop for making surface plots depicting smooth DNN interpolations:

In [ ]:
for k_value, k_group in unique_k_values:
    xb_q2_groups = k_group.groupby(['x_b', 'q_squared'])
    xb_q2_combinations = xb_q2_groups.ngroups

    for (xb_value, qsquared_value), group in xb_q2_groups:

        group = group.sort_values(['t', 'phi'])

        t_values = np.sort(group['t'].unique())
    
        phi_meshgrid, t_meshgrid = np.meshgrid(
            # this is a dense grid of phi values:
            np.linspace(-np.pi, np.pi, 361),
            t_values)

        phi_data = group['phi'].values
        t_data = group['t'].values

        point_dataframe = pd.DataFrame({
            "k": group["k"],
            "q_squared": group["q_squared"],
            "x_b": group["x_b"],
            "t": group["t"],
            "v": np.sin(group["phi"])
        })

        point_predictions_all = np.array([
            predict_bsa(model, point_dataframe, x_scaler, y_scaler)
            for model in models
        ])

        point_mean = np.mean(point_predictions_all, axis = 0)
        point_std = np.std(point_predictions_all, axis = 0)

        bsa_predictions = point_mean.ravel()
        # we don't actually use this:
        # xsec_std = point_std.ravel()

        bsa_actual = group["unp_beam_unp_target_xsec"].to_numpy()

        xsec_residuals = bsa_actual - bsa_predictions

        colors_xsec = np.where(xsec_residuals >= 0, 'red', 'blue')

        surface_dataframe = pd.DataFrame({
            "k": np.full(phi_meshgrid.size, k_value),
            "q_squared": np.full(phi_meshgrid.size, qsquared_value),
            "x_b": np.full(phi_meshgrid.size, xb_value),
            "t": t_meshgrid.ravel(),
            "v": np.sin(phi_meshgrid.ravel())
        })

        surface_predictions_all = np.array([
            predict_bsa(model, surface_dataframe, x_scaler,y_scaler)
            for model in models
        ])

        surface_mean = np.mean(surface_predictions_all, axis = 0)
        surface_std_dev = np.std(surface_predictions_all, axis = 0)
        bsa_surface = surface_mean[:, 0].reshape(phi_meshgrid.shape)
        bsa_stddev_surface = surface_std_dev[:, 0].reshape(phi_meshgrid.shape)

        zero_plane_bsa = np.zeros_like(bsa_surface)

        fig = plt.figure(figsize = (14, 7), layout = "tight")

        ax1 = fig.add_subplot(1, 2, 1, projection = '3d')
        ax2 = fig.add_subplot(1, 2, 2, projection = '3d')

        # [NOTE]: this order actually determines some z-ordering stuff...
        ax1.plot_surface(
            phi_meshgrid, t_meshgrid, bsa_surface + bsa_stddev_surface,
            color = "gray", alpha = 0.30)
        ax1.plot_surface(
            phi_meshgrid, t_meshgrid, bsa_surface - bsa_stddev_surface,
            color = "gray", alpha = 0.30)
        ax1.plot_surface(
            phi_meshgrid, t_meshgrid, bsa_surface,
            cmap = 'viridis', alpha = 0.30)
        ax1.scatter(
            phi_data, t_data, bsa_actual, 
            facecolors = 'white', edgecolors = 'black', s = 20, linewidths = 0.5, alpha = 1.0)

        ax1.set_xlabel(r'$\phi$ [Radians]',
                    labelpad = 16, fontsize = 16.)
        ax1.set_ylabel(r'$t$ [GeV$^{2}$]',
                    labelpad = 16, fontsize = 16.)
        ax1.set_zlabel(r'BSA',
                    labelpad = 7, fontsize = 16.)
        ax1.set_title('BSA', fontsize = 18.)

        ax2.plot_surface(
            phi_meshgrid, t_meshgrid, zero_plane_bsa,
            color = 'gray', alpha = 0.15)
        ax2.scatter(
            phi_data, t_data, xsec_residuals,
            color = colors_xsec, s = 20)

        ax2.set_xlabel(r'$\phi$ [Radians]',
                    labelpad = 16, fontsize = 16.)
        ax2.set_ylabel(r'$t$ [GeV$^{2}$] ',
                    labelpad = 16, fontsize = 16.)
        ax2.set_zlabel('Residuals',
                    labelpad = 7, fontsize = 16.)
        ax2.set_title('BSA Residuals', fontsize = 18)

        fig.suptitle(
            r"DNN Interpolations Across $t$ and $\phi$"
            "\n"
            rf"Kinematic Setting:$k = {k_value}$ GeV, $x_\textrm{{B}} = {xb_value}$, $Q^2 = {qsquared_value}$ GeV$^{{2}}$",
            fontsize = 16.0
        )
        
        for extension in ['png', 'eps']:
            fig.savefig(
                fname = 
                output_directory /
                    f"version_{MAJOR_MINOR_NUMBER}" /
                    "plots" /
                    f"surface_k{k_value}_xb{xb_value}_q2{qsquared_value}_v{MAJOR_MINOR_NUMBER}.{extension}", 
                facecolor = 'white')

        plt.close(fig)